# Chronic Kidney Disease Data Preprocessing
This notebook focuses on loading and preprocessing the Chronic Kidney Disease (CKD) dataset. The main goal is to prepare the raw data for further machine learning tasks by importing the required libraries, locating the dataset, loading the `.arff` file, and checking its structure, data types, missing values, and target class distribution.

In [2]:
# Import regular expression and CSV modules for reading the ARFF file manually
import re
import csv
from pathlib import Path

# Import numerical and data handling libraries
import numpy as np
import pandas as pd

# Import preprocessing and model preparation tools from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Defining Project Paths

In this step, the notebook defines the base, raw, and processed data directories. This helps keep the project structure organized and ensures that processed outputs can be saved in the correct folder.

In [4]:
# Set the main project directory
BASE_DIR = Path.cwd().parents[1]

# Define the paths for raw input data and processed output data
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Create the processed directory if it does not already exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Print directory information for verification
print("Current working directory:", Path.cwd())
print("Base directory:", BASE_DIR)
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Raw exists:", RAW_DIR.exists())
print("Processed exists:", PROCESSED_DIR.exists())


Current working directory: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\preprocessed
Base directory: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction
Raw directory: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\raw
Processed directory: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\processed
Raw exists: True
Processed exists: True


## Checking Raw Data Folder Contents

This step lists the files and folders inside the raw data directory. It is useful for confirming that the required dataset folders are present before loading the CKD data.

In [12]:
# Display all items available inside the raw data folder
print("Raw folder contents:")
for item in RAW_DIR.iterdir():
    print(item.name)

Raw folder contents:
Chronic_Kidney_Disease
data_source.md
Diabetes


## Locating the CKD Dataset File

Here, the full path of the CKD dataset file is defined and checked. This ensures that the `.arff` file exists before attempting to read it.

In [13]:
# Define the file path for the CKD ARFF dataset
ckd_path = RAW_DIR / "Chronic_Kidney_Disease" / "chronic_kidney_disease.arff"

# Print the file path and verify whether the file exists
print("CKD path:", ckd_path)
print("CKD exists:", ckd_path.exists())


CKD path: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\raw\Chronic_Kidney_Disease\chronic_kidney_disease.arff
CKD exists: True


## Creating a Function to Load the ARFF File

Since the CKD dataset is stored in ARFF format, a custom function is created to read the file. The function extracts attribute names and data rows, then converts the dataset into a Pandas DataFrame for easier analysis and preprocessing.

In [15]:
# Define a custom function to load an ARFF file into a pandas DataFrame
def load_ckd_arff(file_path):
    # Store attribute names and data rows
    attributes = []
    rows = []
    in_data_section = False

    # Open the ARFF file safely
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.strip()

            # Skip empty lines
            if not line:
                continue

            low = line.lower()

            # Extract attribute names from the ARFF header
            if low.startswith("@attribute"):
                match = re.match(r"@attribute\s+'?([^']+)'?\s+(.+)$", line, flags=re.IGNORECASE)
                if match:
                    attr_name = match.group(1).strip()
                    attributes.append(attr_name)

            # Detect when the actual data section starts
            elif low.startswith("@data"):
                in_data_section = True

            # Read data rows after the @data section
            elif in_data_section:
                try:
                    parts = next(csv.reader([raw_line], skipinitialspace=True))
                    if len(parts) == len(attributes):
                        rows.append(parts)
                except Exception:
                    continue

    # Return the dataset as a DataFrame
    return pd.DataFrame(rows, columns=attributes)

## Loading the CKD Dataset

The custom function is now used to load the CKD dataset into a DataFrame. After loading, the shape and first few rows are displayed to confirm that the data has been read correctly.

In [16]:
# Load the CKD dataset using the custom ARFF reader
ckd_df = load_ckd_arff(ckd_path)

# Display the number of rows and columns, and preview the dataset
print("CKD shape:", ckd_df.shape)
display(ckd_df.head())

CKD shape: (397, 25)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48,80,1.020,1,0,?,normal,notpresent,notpresent,121,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7,50,1.020,4,0,?,normal,notpresent,notpresent,?,...,38,6000,?,no,no,no,good,no,no,ckd
2,62,80,1.010,2,3,normal,normal,notpresent,notpresent,423,...,31,7500,?,no,yes,no,poor,no,yes,ckd
3,48,70,1.005,4,0,normal,abnormal,present,notpresent,117,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51,80,1.010,2,0,normal,normal,notpresent,notpresent,106,...,35,7300,4.6,no,no,no,good,no,no,ckd


## Displaying Column Names

This step prints all column names in the CKD dataset. It helps in understanding the structure of the dataset and identifying input features and target variables.

In [17]:
# Print all column names in the CKD dataset
print("CKD columns:")
print(ckd_df.columns.tolist())

CKD columns:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']


## Checking Data Types and Missing Values

In this step, the data types of all features are examined, and missing values are counted. This is an important preprocessing step because it helps identify columns that may require type conversion, cleaning, or imputation.

In [18]:
# Display the data types of each column
print("CKD data types:")
print(ckd_df.dtypes)

# Count and display missing values in each column
print("\nCKD missing values:")
print(ckd_df.isnull().sum().sort_values(ascending=False))

# Print a separator line for cleaner notebook output
print("\n" + "="*60)

CKD data types:
age      object
bp       object
sg       object
al       object
su       object
rbc      object
pc       object
pcc      object
ba       object
bgr      object
bu       object
sc       object
sod      object
pot      object
hemo     object
pcv      object
wbcc     object
rbcc     object
htn      object
dm       object
cad      object
appet    object
pe       object
ane      object
class    object
dtype: object

CKD missing values:
age      0
bp       0
sg       0
al       0
su       0
rbc      0
pc       0
pcc      0
ba       0
bgr      0
bu       0
sc       0
sod      0
pot      0
hemo     0
pcv      0
wbcc     0
rbcc     0
htn      0
dm       0
cad      0
appet    0
pe       0
ane      0
class    0
dtype: int64



## Exploring the Target Variable

The target column for the CKD dataset is identified as `class`. The distribution of this target variable is then displayed to understand how many records belong to each class and whether the dataset is balanced or imbalanced.

In [19]:
# Define the target variable for the CKD dataset
ckd_target = "class"

# Display the class distribution of the target column
print("CKD target distribution:")
print(ckd_df[ckd_target].value_counts(dropna=False))

CKD target distribution:
class
ckd       246
notckd    149
ckd\t       2
Name: count, dtype: int64
